# 🫁 Pneumonia Classifier — Train on Colab

Train a DenseNet121 to classify chest X-rays as pneumonia / no-pneumonia.

**⚠️ Disclaimer:** Educational/research only. Not for clinical use.

## Setup
1. Runtime → Change runtime type → **T4 GPU** (free) or A100 (Pro)
2. Run all cells in order
3. When prompted, upload `kaggle.json` and paste WandB key

Total time: ~30 minutes on T4 GPU.

## 1. Verify GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ No GPU — go to Runtime → Change runtime type → T4 GPU')

## 2. Install dependencies

In [ ]:
!pip install --quiet monai pandas scikit-learn tqdm matplotlib wandb gradio grad-cam kaggle

## 3. Clone code repository

In [ ]:
!git clone https://github.com/aommiez/medical-ai-pneumonia.git /content/medical-ai
%cd /content/medical-ai
!ls

## 4. Setup Kaggle credentials

Upload your `kaggle.json` file (from https://www.kaggle.com/settings → API → Create New Token).

In [ ]:
from google.colab import files
import os
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

## 5. Download dataset (~2.4 GB)

In [ ]:
!mkdir -p data/raw && cd data/raw && kaggle datasets download -d paultimothymooney/chest-xray-pneumonia --unzip
!ls data/raw/chest_xray/

## 6. WandB login (paste your API key when prompted)

In [ ]:
import wandb
wandb.login()  # paste key from https://wandb.ai/authorize

## 7. Build train/val/test manifests from folder structure

Kaggle subset uses folder layout; our `src/dataset.py` expects a CSV. Create one.

In [ ]:
import pandas as pd
from pathlib import Path

root = Path('data/raw/chest_xray')
rows = []
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        for img in (root / split / cls).glob('*.jpeg'):
            rows.append({
                'split': split,
                'Image Index': str(img.relative_to(root)),
                'Finding Labels': 'Pneumonia' if cls == 'PNEUMONIA' else 'No Finding',
            })
df = pd.DataFrame(rows)
print(df.groupby(['split','Finding Labels']).size())
for split in ['train','val','test']:
    df[df.split == split].to_csv(f'data/{split}.csv', index=False)
    print(f'wrote data/{split}.csv: {(df.split==split).sum()} rows')

## 8. Train

Uses `src/train.py` from the repo. ~10 min/epoch on T4, so 30 epochs ≈ 5 hours. Reduce `--epochs` to 10 for first try (~50 minutes).

In [ ]:
!python src/train.py \
    --data-root data/raw/chest_xray \
    --train-csv data/train.csv \
    --val-csv data/val.csv \
    --epochs 10 \
    --batch-size 32 \
    --lr 1e-4 \
    --device cuda \
    --output checkpoints/best.pt

## 9. Evaluate on test set

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from monai.transforms import Compose, EnsureType, Resize, ScaleIntensity, ToTensor
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

import sys; sys.path.insert(0, '.')
from src.model import build_model
from src.dataset import NIHPneumoniaDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model(num_classes=1, pretrained=False).to(device)
model.load_state_dict(torch.load('checkpoints/best.pt', map_location=device))
model.eval()

tfm = Compose([Resize((224,224)), ScaleIntensity(), ToTensor(), EnsureType()])
test_ds = NIHPneumoniaDataset('data/raw/chest_xray', 'data/test.csv', tfm)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

ys, ps = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        p = torch.sigmoid(model(x)).cpu().numpy().squeeze()
        ys.extend(y.numpy().squeeze().tolist())
        ps.extend(p.tolist())

auroc = roc_auc_score(ys, ps)
preds = (np.array(ps) > 0.5).astype(int)
cm = confusion_matrix(ys, preds)
print(f'Test AUROC: {auroc:.4f}')
print(f'Confusion matrix:\n{cm}')
print(classification_report(ys, preds, target_names=['No Pneumonia', 'Pneumonia']))

## 10. Plot ROC curve

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(ys, ps)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'AUROC = {auroc:.3f}')
plt.plot([0,1],[0,1],'k--',alpha=0.3)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC — Pneumonia Classifier')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('docs/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save checkpoint to Google Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/medical-ai/checkpoints
!cp checkpoints/best.pt /content/drive/MyDrive/medical-ai/checkpoints/
!cp docs/roc_curve.png /content/drive/MyDrive/medical-ai/ 2>/dev/null || true
print('Saved to Drive: medical-ai/checkpoints/best.pt')

## 12. Download checkpoint to your machine

In [ ]:
from google.colab import files
files.download('checkpoints/best.pt')
files.download('docs/roc_curve.png')

## Next steps

After training:
1. Update `README.md` with your AUROC + ROC plot
2. Deploy demo on Hermes VPS: `scp checkpoints/best.pt hermes@<vps>:~/medical-ai/checkpoints/`
3. Run `python demo/app.py` on VPS + expose via Cloudflare Tunnel
4. Add link to portfolio

See `docs/next_steps.md` for full deployment guide.